In [ ]:
!pip install -q transformers datasets accelerate sentencepiece protobuf
!pip install -q gensim scikit-learn pandas openpyxl regex torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 82.4 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import regex as re
import json, os, gc, time
from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
import torch
from transformers import (
    T5ForConditionalGeneration,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
from torch.utils.data import Dataset
import warnings
warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42
MIN_CLASS_COUNT = 20
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/FactGrid Cuneiform (AWCA)/people/Thompson/'

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB
Mounted at /content/drive


In [ ]:
POS_UNIFIED_MAP = {
    'akk': {
        'N': 'NOUN', 'V': 'VERB', 'AJ': 'ADJ', 'AV': 'ADV',
        'PRP': 'ADP', 'DET': 'DET', 'CNJ': 'CONJ', 'MOD': 'MOD',
        'REL': 'REL', 'SBJ': 'SBJN', 'IP': 'INTJ',
        'PN': 'PN', 'DN': 'DN', 'GN': 'GN', 'CN': 'CN',
        'RN': 'RN', 'QN': 'QN', 'WN': 'WN', 'MN': 'MN',
        'AN': 'AN', 'FN': 'FN', 'TN': 'TN', 'LN': 'LN', 'ON': 'ON',
        'n': 'NUM', 'u': 'X', 'X': 'X',
    },
    'sux': {
        'N': 'NOUN', 'V/t': 'VERB', 'V/i': 'VERB', 'V': 'VERB',
        'AJ': 'ADJ', 'AV': 'ADV',
        'NU': 'NUM', 'IP': 'INTJ', 'QP': 'QP',
        'DN': 'DN', 'GN': 'GN', 'PN': 'PN', 'RN': 'RN',
        'SN': 'SN', 'TN': 'TN', 'WN': 'WN', 'MN': 'MN',
        'NA': 'X',
    },
    'elx': {
        'Noun': 'NOUN', 'Verb': 'VERB', 'ADJ': 'ADJ', 'other': 'OTHER',
        'PN': 'PN', 'PN-hyp': 'PN', 'GN': 'GN', 'DN': 'DN', 'Magic': 'OTHER',
    },
}

ENTITY_TAGS = {'PN', 'DN', 'GN', 'CN', 'RN', 'QN', 'WN', 'MN', 'AN', 'FN', 'TN', 'LN', 'ON', 'SN', 'QP'}

def map_pos_unified(pos_raw, lang):
    return POS_UNIFIED_MAP.get(lang, {}).get(pos_raw, 'X')

def map_pos_grammatical(pos_unified):
    return 'PROPN' if pos_unified in ENTITY_TAGS else pos_unified

def map_ner_tag(pos_unified):
    return pos_unified if pos_unified in ENTITY_TAGS else 'O'

print("POS harmonization maps loaded.")

POS harmonization maps loaded.


In [ ]:
sign_list = pd.read_json(
    'https://raw.githubusercontent.com/situx/Nuolenna/master/sign_list.json', orient='index'
)
sign_list.columns = ['unicode']
sign_list['sign'] = sign_list.index.tolist()
sign_list = sign_list[['sign', 'unicode']].reset_index(drop=True)

akkademia = pd.read_csv(
    'https://raw.githubusercontent.com/gaigutherz/Akkademia/master/cuneiform_to_unicode_fixed.csv'
)
merged = pd.merge(sign_list, akkademia, on=['sign', 'unicode'], how='outer')
sign_dict = dict(zip(merged['sign'].astype(str), merged['unicode'].astype(str)))

# Load manual corrections
UNMATCHED_PATH_1 = BASE_PATH + 'unmatchednew_AAedit - unmatchednew.csv'
UNMATCHED_PATH_2 = BASE_PATH + 'unmatchednew - solonew.csv'
try:
    unmatched = pd.read_csv(UNMATCHED_PATH_1)[['unmatched_sign','use']].dropna()
    unmatched2 = pd.read_csv(UNMATCHED_PATH_2)[['value', 'SIGN']].dropna()
    manual_dict = dict(zip(unmatched['unmatched_sign'], unmatched['use']))
    manual_dict2 = dict(zip(unmatched2['value'].str.strip("[]' "), unmatched2['SIGN']))
    sign_dict.update(manual_dict)
    sign_dict.update(manual_dict2)
    print(f"Manual corrections loaded: {len(manual_dict)} + {len(manual_dict2)}")
except FileNotFoundError:
    print("Manual correction files not found — using base sign lists only.")

print(f"Total sign mappings: {len(sign_dict)}")

Manual corrections loaded: 114 + 22
Total sign mappings: 18271


In [ ]:
def normalize_transliteration(text, lang='akk'):
    if pd.isna(text) or text == '':
        return ''
    s = str(text)
    s = re.sub(r'\{[^}]*\}', '', s)
    s = s.replace('-', ' ').replace('.', ' ')
    for ch in ['[', ']', '#', '!', '?', '*', '(', ')']:
        s = s.replace(ch, '')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def transliteration_to_unicode(text, sign_dict, lang='akk'):
    normalized = normalize_transliteration(text, lang)
    if not normalized:
        return '', []
    tokens = normalized.split()
    out, unmatched = [], []
    for tok in tokens:
        if tok in sign_dict:
            out.append(sign_dict[tok])
        elif tok.lower() in sign_dict:
            out.append(sign_dict[tok.lower()])
        elif tok.upper() in sign_dict:
            out.append(sign_dict[tok.upper()])
        else:
            out.append(tok)
            if re.search(r'\p{Latin}', tok):
                unmatched.append(tok)
    return ' '.join(out), unmatched

def convert_column_to_unicode(forms, sign_dict, lang='akk'):
    results = forms.apply(lambda x: transliteration_to_unicode(x, sign_dict, lang))
    unicode_col = results.apply(lambda x: x[0])
    unmatched_col = results.apply(lambda x: x[1])
    n_clean = (unmatched_col.apply(len) == 0).sum()
    rate = n_clean / len(forms) if len(forms) > 0 else 0
    return unicode_col, rate

print("Unicode conversion functions defined.")

Unicode conversion functions defined.


In [ ]:
def load_akkadian(csv_path):
    df = pd.read_csv(csv_path, low_memory=False)
    df = df[df['pos'].notna() & (df['pos'] != 'u')].copy()
    df = df[df['form'].notna() & (df['form'] != 'x') & (df['form'] != 'X')].copy()
    out = pd.DataFrame({
        'token_id': range(len(df)),
        'text_id': df['id_text'].values,
        'form_latin': df['form'].values,
        'pos_raw': df['pos'].values,
        'lemma': df['cf'].values,
        'gloss': df['gw'].values,
        'language': 'akk',
    })
    out['pos_unified'] = out['pos_raw'].apply(lambda x: map_pos_unified(x, 'akk'))
    out['pos_grammatical'] = out['pos_unified'].apply(map_pos_grammatical)
    out['ner_tag'] = out['pos_unified'].apply(map_ner_tag)
    return out

def load_sumerian(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df['pos'].notna() & (df['pos'] != '') & (df['pos'].astype(str) != 'nan')].copy()
    df = df[df['form'].notna()].copy()
    out = pd.DataFrame({
        'token_id': range(len(df)),
        'text_id': df['id_text'].values,
        'form_latin': df['form'].values,
        'pos_raw': df['pos'].values,
        'lemma': df['cf'].values,
        'gloss': df['gw'].values,
        'language': 'sux',
    })
    out['pos_unified'] = out['pos_raw'].apply(lambda x: map_pos_unified(x, 'sux'))
    out['pos_grammatical'] = out['pos_unified'].apply(map_pos_grammatical)
    out['ner_tag'] = out['pos_unified'].apply(map_ner_tag)
    return out

def add_unicode_representations(df, sign_dict):
    df['form_unicode'], rate = convert_column_to_unicode(
        df['form_latin'], sign_dict, df['language'].iloc[0]
    )
    df['form_unicode_nospace'] = df['form_unicode'].str.replace(' ', '')
    df['unicode_clean'] = ~df['form_unicode'].apply(
        lambda x: bool(re.search(r'\p{Latin}', str(x))) if pd.notna(x) else True
    )
    print(f"  Unicode conversion rate: {rate:.1%} ({df['unicode_clean'].sum()}/{len(df)})")
    return df

print("Data loaders defined.")

Data loaders defined.


In [ ]:
AKK_PATH = BASE_PATH + 'alltexts_AKK.csv'
SUX_PATH = BASE_PATH + 'alltexts_SUX.csv'

print("Loading Akkadian...")
akk = load_akkadian(AKK_PATH)
print(f"  {len(akk):,} tokens, {akk['text_id'].nunique():,} texts")

print("\nLoading Sumerian...")
sux = load_sumerian(SUX_PATH)
print(f"  {len(sux):,} tokens, {sux['text_id'].nunique():,} texts")

print("\nConverting Akkadian to Unicode...")
akk = add_unicode_representations(akk, sign_dict)

print("\nConverting Sumerian to Unicode...")
sux = add_unicode_representations(sux, sign_dict)

datasets = {'akk': akk, 'sux': sux}

Loading Akkadian...
  1,255,669 tokens, 13,720 texts

Loading Sumerian...
  146,143 tokens, 394 texts

Converting Akkadian to Unicode...
  Unicode conversion rate: 98.9% (1232012/1255669)

Converting Sumerian to Unicode...
  Unicode conversion rate: 99.9% (145217/146143)


In [ ]:
DICT_PATH = BASE_PATH + 'Elamite_Lemma-base-draft.xlsx'
NASU_PATH = BASE_PATH + 'UnTN-Nasu texts Word-level.csv'

# 1. Load dictionary from Excel tabs
all_sheets = pd.read_excel(DICT_PATH, sheet_name=None)
target_tabs = ['ADJ', 'Noun', 'Verb', 'other', 'PN', 'PN-hyp', 'GN', 'DN', 'Magic']
columns_to_keep = [
    'transliteration', 'sorting', 'period', 'base',
    'logogram', 'morpheme_1', 'morpheme_2', 'morpheme_3',
    'sense_hk', 'certainty-weight_hk', 'sense_hk_qid',
    'certainty-weight_MEGA', 'sense_MEGA_qid', 'POS', 'number', 'person'
]
combined_list = []
for name in target_tabs:
    if name in all_sheets:
        df = all_sheets[name]
        df['category'] = name
        existing_cols = [c for c in columns_to_keep if c in df.columns]
        combined_list.append(df[existing_cols + ['category']])
final_dictionary = pd.concat(combined_list, ignore_index=True)
print(f"Dictionary: {len(final_dictionary)} entries from {len(combined_list)} tabs")

# 2. First-pass preprocessing
to_unicode = final_dictionary.copy()
to_unicode['transliteration original'] = to_unicode['transliteration']
to_unicode['transliteration'] = to_unicode['transliteration'].astype(str).fillna('')
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('-', ' ')
for ch in ['_', '[', ']', '*', '!', '?', '/', ',', ':', ';', '^', '`']:
    to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(ch, '', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('.', ' ', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.lower()
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace(r'~.*?……', '……', regex=True)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('X', '', regex=False)
to_unicode['transliteration'] = to_unicode['transliteration'].str.replace('……', ' ', regex=False)

def replace_with_unicode(text):
    return ' '.join([sign_dict.get(s, s) for s in text.split()])

to_unicode['unicode'] = to_unicode['transliteration'].apply(replace_with_unicode)
to_unicode['roman'] = to_unicode['unicode'].apply(
    lambda x: any(bool(re.search(r'\p{Latin}', w)) for w in x.split()))
print(f"After first pass: {(~to_unicode['roman']).sum()}/{len(to_unicode)} clean "
      f"({(~to_unicode['roman']).sum()/len(to_unicode)*100:.1f}%)")

# 3. Second-pass conversion
unicode_dict_v2 = dict(zip(merged['sign'].astype(str), merged['unicode'].astype(str)))
unicode_dict_v2.update(manual_dict)
unicode_dict_v2.update(manual_dict2)

def normalize_for_unmatched_pass(s):
    if pd.isna(s): return ""
    s = str(s)
    s = re.sub(r"\(\s*md\s*\)", " m d ", s)
    s = re.sub(r"[.,:;!?()\[\]{}<>\\\"\"\"''/\\|*^`~]", " ", s)
    s = re.sub(r"[-\u2013\u2014]", " ", s)
    s = re.sub(r"[₀-₉]+", "", s)
    s = re.sub(r"\d+", "", s)
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def second_pass_unicode(text):
    tokens = normalize_for_unmatched_pass(text).split()
    return ' '.join([unicode_dict_v2.get(t, t) for t in tokens])

still_roman = to_unicode[to_unicode['roman']].copy()
still_roman['unicode_v2'] = still_roman['transliteration original'].apply(second_pass_unicode)
still_roman['roman_v2'] = still_roman['unicode_v2'].apply(
    lambda x: any(bool(re.search(r'\p{Latin}', w)) for w in x.split()))
to_unicode.loc[still_roman.index, 'unicode'] = still_roman['unicode_v2']
to_unicode.loc[still_roman.index, 'roman'] = still_roman['roman_v2']
print(f"After second pass: {(~to_unicode['roman']).sum()}/{len(to_unicode)} clean "
      f"({(~to_unicode['roman']).sum()/len(to_unicode)*100:.1f}%)")

# 4. Build Elamite dataframe
elx_clean = to_unicode[~to_unicode['roman']].copy()
elx = pd.DataFrame({
    'token_id': range(len(elx_clean)),
    'text_id': 'dict_' + elx_clean.index.astype(str),
    'form_latin': elx_clean['transliteration'].values,
    'form_unicode': elx_clean['unicode'].values,
    'pos_raw': elx_clean['category'].values,
    'language': 'elx',
})
elx['pos_unified'] = elx['pos_raw'].apply(lambda x: map_pos_unified(x, 'elx'))
elx['pos_grammatical'] = elx['pos_unified'].apply(map_pos_grammatical)
elx['ner_tag'] = elx['pos_unified'].apply(map_ner_tag)
print(f"\nElamite: {len(elx):,} clean entries")
print(f"POS: {elx['pos_unified'].value_counts().to_dict()}")

datasets['elx'] = elx
print(f"\nAll datasets loaded: {list(datasets.keys())}")
for lang, df in datasets.items():
    print(f"  {lang.upper()}: {len(df):,} tokens")

Dictionary: 15601 entries from 8 tabs
After first pass: 13302/15601 clean (85.3%)
After second pass: 13322/15601 clean (85.4%)

Elamite: 13,322 clean entries
POS: {'PN': 4980, 'NOUN': 3847, 'VERB': 1499, 'GN': 1475, 'OTHER': 683, 'ADJ': 538, 'DN': 300}

All datasets loaded: ['akk', 'sux', 'elx']
  AKK: 1,255,669 tokens
  SUX: 146,143 tokens
  ELX: 13,322 tokens


In [ ]:
for lang, df in datasets.items():
    if 'entity_detect' not in df.columns:
        df['entity_detect'] = df['pos_unified'].apply(
            lambda x: 'ENTITY' if x in ENTITY_TAGS else 'NON-ENTITY'
        )
    if 'entity_type' not in df.columns:
        df['entity_type'] = df['pos_unified'].apply(
            lambda x: x if x in ENTITY_TAGS else 'NON-ENTITY'
        )
    datasets[lang] = df

print("Entity task columns added.")

Entity task columns added.


In [ ]:
class CuneiformT5Dataset(Dataset):
    """Input: "classify: <text>" → Target: "<POS_label>" """
    def __init__(self, texts, labels, tokenizer, max_input_len=128, max_target_len=16):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        input_text = f"classify: {self.texts[idx]}"
        target_text = str(self.labels[idx])

        input_enc = self.tokenizer(
            input_text, max_length=self.max_input_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        target_enc = self.tokenizer(
            target_text, max_length=self.max_target_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )

        labels = target_enc.input_ids.squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_enc.input_ids.squeeze(),
            'attention_mask': input_enc.attention_mask.squeeze(),
            'labels': labels,
        }

In [ ]:
def train_byt5(texts, labels, tokenizer, fold_name="", n_epochs=10,
               batch_size=16, lr=3e-4, max_input_len=128):
    """Fine-tune ByT5-small with 3-fold CV. Returns mean F1."""
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(texts, labels)):
        print(f"      Fold {fold+1}/3...", end=" ", flush=True)

        train_texts = [texts[i] for i in train_idx]
        train_labels = [labels[i] for i in train_idx]
        val_texts = [texts[i] for i in val_idx]
        val_labels = [labels[i] for i in val_idx]

        train_ds = CuneiformT5Dataset(train_texts, train_labels, tokenizer, max_input_len)
        val_ds = CuneiformT5Dataset(val_texts, val_labels, tokenizer, max_input_len)

        model = T5ForConditionalGeneration.from_pretrained('google/byt5-small')
        model = model.to(device)

        training_args = TrainingArguments(
            output_dir=f'/content/byt5_{fold_name}_fold{fold}',
            num_train_epochs=n_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            learning_rate=lr,
            weight_decay=0.01,
            warmup_ratio=0.1,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='eval_loss',
            save_total_limit=1,
            fp16=True,
            logging_steps=100,
            report_to='none',
            seed=SEED,
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        trainer.train()

        # Generate predictions
        model.eval()
        predictions = []
        for i in range(0, len(val_texts), batch_size * 2):
            batch_texts = val_texts[i:i + batch_size * 2]
            inputs = tokenizer(
                [f"classify: {t}" for t in batch_texts],
                max_length=max_input_len, padding=True,
                truncation=True, return_tensors='pt'
            ).to(device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=16,
                    num_beams=1, do_sample=False,
                )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(decoded)

        predictions_clean = [p.strip() for p in predictions]
        f1 = f1_score(val_labels, predictions_clean, average='macro', zero_division=0)
        fold_scores.append(f1)
        print(f"F1={f1:.4f}")

        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    return mean_f1, std_f1

In [ ]:
print("Loading ByT5 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained('google/byt5-small')

TASKS = [
    ('pos_unified', 'unified_pos'),
    ('pos_grammatical', 'gram_pos'),
    ('entity_detect', 'ent_detect'),
    ('entity_type', 'ent_type'),
]

MAX_SAMPLES = 5000
byt5_results = {}

for lang, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"  ByT5 — {lang.upper()}")
    print(f"{'='*60}")

    byt5_results[lang] = {}

    for pos_col, task_name in TASKS:
        if pos_col not in df.columns:
            print(f"\n  Skipping {task_name} — column {pos_col} not found")
            continue

        counts = df[pos_col].value_counts()
        valid = counts[counts >= MIN_CLASS_COUNT].index.tolist()
        task_df = df[df[pos_col].isin(valid)].copy()

        if len(task_df) > MAX_SAMPLES:
            task_df = task_df.sample(MAX_SAMPLES, random_state=SEED)

        if len(valid) < 2:
            print(f"\n  Skipping {task_name} — fewer than 2 valid classes")
            continue

        texts_l = task_df['form_latin'].astype(str).tolist()
        texts_u = task_df['form_unicode'].astype(str).tolist()
        texts_c = [f"{l} | {u}" for l, u in zip(texts_l, texts_u)]
        labels = task_df[pos_col].astype(str).tolist()

        print(f"\n  {task_name} ({len(task_df):,} tokens, {len(valid)} classes)")

        # Latin
        print(f"    Latin:")
        f1_l, std_l = train_byt5(texts_l, labels, tokenizer,
                                  f"{lang}_{task_name}_latin")
        print(f"    → {f1_l:.4f} (±{std_l:.4f})")

        # Unicode
        print(f"    Unicode:")
        f1_u, std_u = train_byt5(texts_u, labels, tokenizer,
                                  f"{lang}_{task_name}_unicode")
        print(f"    → {f1_u:.4f} (±{std_u:.4f})")

        # Concat
        print(f"    Concat:")
        f1_c, std_c = train_byt5(texts_c, labels, tokenizer,
                                  f"{lang}_{task_name}_concat",
                                  max_input_len=256)
        print(f"    → {f1_c:.4f} (±{std_c:.4f})")

        gain = f1_c - max(f1_l, f1_u)
        print(f"    Concat gain: {gain:+.4f} {'***' if gain > 0.01 else ('+' if gain > 0 else '-')}")

        byt5_results[lang][task_name] = {
            'latin': f1_l, 'unicode': f1_u, 'concat': f1_c, 'gain': gain
        }

        # Save checkpoint after every task
        with open('/content/byt5_results_checkpoint.json', 'w') as f:
            json.dump(byt5_results, f, indent=2)
        print(f"    [Checkpoint saved]")

Loading ByT5 tokenizer...


config.json:   0%|          | 0.00/698 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]


  ByT5 — AKK

  unified_pos (5,000 tokens, 24 classes)
    Latin:
      Fold 1/3... 

...

Epoch,Training Loss,Validation Loss
1,0.000000,nan
2,0.000000,nan
3,0.000000,nan
4,0.000000,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print(f"\n{'='*60}")
print(f"  FULL ARCHITECTURE COMPARISON")
print(f"{'='*60}")

lr = {
    'akk': {'unified_pos': {'l': .693, 'u': .621, 'c': .712},
            'gram_pos':    {'l': .669, 'u': .657, 'c': .682},
            'ent_detect':  {'l': .937, 'u': .839, 'c': .939},
            'ent_type':    {'l': .897, 'u': .833, 'c': .910}},
    'sux': {'unified_pos': {'l': .847, 'u': .750, 'c': .870},
            'gram_pos':    {'l': .872, 'u': .832, 'c': .888},
            'ent_detect':  {'l': .947, 'u': .896, 'c': .954},
            'ent_type':    {'l': .957, 'u': .923, 'c': .963}},
    'elx': {'unified_pos': {'l': .636, 'u': .659, 'c': .666},
            'gram_pos':    {'l': .615, 'u': .634, 'c': .642},
            'ent_detect':  {'l': .873, 'u': .876, 'c': .882},
            'ent_type':    {'l': .871, 'u': .881, 'c': .883}},
}

tf = {
    'akk': {'unified_pos': {'l': .360, 'u': .373, 'c': .412},
            'gram_pos':    {'l': .516, 'u': .470, 'c': .513}},
    'sux': {'unified_pos': {'l': .380, 'u': .428, 'c': .451},
            'gram_pos':    {'l': .561, 'u': .630, 'c': .654}},
    'elx': {'unified_pos': {'l': .446, 'u': .471, 'c': .489},
            'gram_pos':    {'l': .430, 'u': .465, 'c': .469}},
}

print(f"\n  {'Task':<15s} {'Lang':<5s} | {'--- LR ---':^21s} | {'- Transf. -':^21s} | {'--- ByT5 ---':^21s}")
print(f"  {'':15s} {'':5s} | {'L':>6s} {'U':>6s} {'C':>6s} | {'L':>6s} {'U':>6s} {'C':>6s} | {'L':>6s} {'U':>6s} {'C':>6s}")
print(f"  {'-'*15} {'-'*5} + {'-'*21} + {'-'*21} + {'-'*21}")

concat_wins_byt5 = 0
total_byt5 = 0

for lang in ['akk', 'sux', 'elx']:
    if lang not in byt5_results:
        continue
    for task in ['unified_pos', 'gram_pos', 'ent_detect', 'ent_type']:
        if task not in byt5_results.get(lang, {}):
            continue

        bt = byt5_results[lang][task]
        lr_t = lr.get(lang, {}).get(task, {})
        tf_t = tf.get(lang, {}).get(task, {})

        lr_str = f"{lr_t.get('l',0):>6.3f} {lr_t.get('u',0):>6.3f} {lr_t.get('c',0):>6.3f}" if lr_t else f"{'---':>6s} {'---':>6s} {'---':>6s}"
        tf_str = f"{tf_t.get('l',0):>6.3f} {tf_t.get('u',0):>6.3f} {tf_t.get('c',0):>6.3f}" if tf_t else f"{'---':>6s} {'---':>6s} {'---':>6s}"
        bt_str = f"{bt['latin']:>6.3f} {bt['unicode']:>6.3f} {bt['concat']:>6.3f}"

        total_byt5 += 1
        if bt['gain'] > 0:
            concat_wins_byt5 += 1

        print(f"  {task:<15s} {lang.upper():<5s} | {lr_str} | {tf_str} | {bt_str}")

print(f"\n  Concat wins across architectures:")
print(f"    LR (n-gram):        11/12")
print(f"    Char Transformer:    5/6")
print(f"    ByT5 (pretrained):   {concat_wins_byt5}/{total_byt5}")

if concat_wins_byt5 == total_byt5:
    print(f"\n  → Complementarity persists across ALL architectures including PLMs!")
elif concat_wins_byt5 > total_byt5 // 2:
    print(f"\n  → Complementarity mostly persists under PLMs.")
else:
    print(f"\n  → PLM pretraining may reduce the need for explicit dual representation.")